# 2. PD model

Builds the probability of default (PD) model and its scorecard.

**Input:** `data/loan_data.pkl`.  
**Output:** PD predictions for each loan and the scorecard.

**2.1 Load the data  
2.2 Target and inputs  
2.3 Train/test split  
2.4 Prepare the inputs  
2.5 Model  
2.6 Evaluation  
2.7 Scorecard  
2.8 Save**

## 2.1 Load the data

We read `loan_data.pkl`, the table created in notebook 01, and check that it is the same table: the number of rows and columns and the default rate.

In [1]:
import numpy as np
import pandas as pd

data_dir = '../data'  # relative to the notebooks folder

In [2]:
loan_data = pd.read_pickle(f'{data_dir}/loan_data.pkl')
loan_data.shape
# We expect: (244967, 31)

(244967, 31)

In [3]:
loan_data['default'].mean()
# We expect ~ 0.0575

np.float64(0.057493458302546874)

In [4]:
#loan_data.head()

## 2.2 Target and inputs

**Target.** The target is `default`: 1 if the loan defaulted, 0 otherwise. The model estimates the probability of default, PD = P(`default` = 1).

**Inputs.** The inputs are the loan and borrower characteristics known at origination. We exclude the following columns:

| Column | Reason |
|---|---|
| `default` | The target. |
| `loan_id` | Identifier of the loan; it does not describe the loan. |
| `zero_balance_code`, `zero_balance_removal_upb`, `actual_loss` | Known only when the loan closes, so they would leak the outcome. They are used in the LGD and EAD models. |
| `vintage`, `first_payment_date` | They identify the origination period. The model would learn the economic conditions of a period, not a characteristic of the borrower, and this does not carry over to new loans. |
| `maturity_date` | Equals `first_payment_date` plus `orig_loan_term`. It adds no information beyond the loan term. |
| `pre_harp_loan_id` | Identifier of the refinanced loan, filled only for HARP loans. The indicator `harp_indicator` already contains this information. |

In [5]:
input_cols = ['credit_score', 'first_time_homebuyer', 'msa', 'mi_pct', 'num_units',
              'occupancy_status', 'cltv', 'dti', 'orig_upb', 'ltv', 'int_rate',
              'channel', 'prepay_penalty', 'property_state', 'property_type',
              'postal_code', 'loan_purpose', 'orig_loan_term', 'num_borrowers',
              'seller_name', 'super_conforming', 'harp_indicator']
# The columns we keep as inputs

len(input_cols)

22

In [6]:
loan_data_inputs = loan_data[input_cols].copy()
loan_data_inputs.shape
#loan_data_inputs.head()

(244967, 22)

In [7]:
loan_data_targets = loan_data['default'].copy()
loan_data_targets.shape
#loan_data_targets.head()

(244967,)

In [8]:
loan_data_targets.mean()
# We expect ~ 0.575

np.float64(0.057493458302546874)

## 2.3 Train/test split

We split the loans randomly into a training set (80%) and a test set (20%). The model is fitted on the training set only. The test set is used once, at the end, to evaluate the model on loans it has not seen. The split is stratified by `default`, so both sets have the same default rate. The random seed is fixed so that the split is reproducible.

We split before preparing the inputs, so that nothing learned from the data in the next sections uses the test set.

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
loan_data_inputs_train, loan_data_inputs_test, loan_data_targets_train, loan_data_targets_test = train_test_split(
    loan_data_inputs, loan_data_targets, test_size = 0.2, random_state = 42, stratify = loan_data_targets)
# We split the data

In [11]:
print(loan_data_inputs_train.shape, loan_data_inputs_test.shape)
print(loan_data_targets_train.shape, loan_data_targets_test.shape)
print(loan_data_targets_train.mean(), loan_data_targets_test.mean())

(195973, 22) (48994, 22)
(195973,) (48994,)
0.05749261377842866 0.05749683634730783
